### Function
- `train_test_split()` = ฟังก์ชันสุ่มแบ่งข้อมูลออกเป็นชุดฝึก และ ชุดทดสอบ  
- `.fit()` = เมธอดสั่งให้โมเดล Machine Learning เรียนรู้จากชุดฝึก
- `.predict` = เมธอดนำโมเดลไปทำนายข้อมูลนักเตะชุดใหม่
- `mean_absolute_error()` = ฟังก์ชันคำนวณค่าความผิดพลากเฉลี่ย (MAE) ระหว่างคำตอบจริงกับค่าหาย

### Building Machine Learning Models

1. Data Preparation
2. Spliting feature (overall, age, pace, shooting) and target (value_millions) ==> DataFrame 20
3. Spliting Training and Testing (60:40, 70:30)
4. Training model
5. MAE Evaluation
6. Prediction player target 

In [159]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


In [160]:
# Data Preparation
df = pd.read_csv("../data/footballers.csv")
display(df.head(3))


,name,club,league,nationality,position,age,height_cm,overall,pace,shooting,passing,dribbling,defending,physical,value_millions
0,Lionel Messi,Inter Miami,MLS,Argentina,RW,37.0,170.0,90,87,90,91,94,34,65,45
1,Cristiano Ronaldo,Al Nassr,Saudi Pro League,Portugal,ST,39.0,187.0,86,78,89,82,83,32,75,15
2,Kylian Mbappe,Real Madrid,La Liga,France,ST,26.0,178.0,91,97,88,80,92,36,78,180


In [161]:
# Cleaning

df["value_millions"] = pd.to_numeric(df["value_millions"], errors="coerce")
df_clean = df.dropna(subset=["value_millions", "overall", "age", "pace", "shooting"]).copy()

print(f"Data sample for training model: {len(df_clean)} individuals📊")
display(df_clean[["name", "overall", "age", "pace", "shooting", "value_millions"]].head())

Data sample for training model: 44 individuals📊


,name,overall,age,pace,shooting,value_millions
0,Lionel Messi,90,37.0,87,90,45.0
1,Cristiano Ronaldo,86,39.0,78,89,15.0
2,Kylian Mbappe,91,26.0,97,88,180.0
3,Erling Haaland,91,24.0,89,91,170.0
4,Kevin De Bruyne,88,33.0,76,89,60.0


In [162]:
# Train / Testing

# 1.1 Define Features (2D) and Target 1D
feature_cols = ["overall", "age", "pace", "shooting"]
x = df_clean[feature_cols]
y = df_clean["value_millions"]

# 1.2 Split Train for 70% / Test for 30%
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.3, random_state=42)

print(f"Size of training (Train Set): {X_train.shape[0]} rows")
print(f"Size of testing (Test Set): {Y_test.shape[0]} rows")

Size of training (Train Set): 30 rows
Size of testing (Test Set): 14 rows


In [163]:
# Training Model with LinearRegression

# 2.1 Creating baseline model (to predict wiht avg Y_train)
baseline_prediction = [Y_train.mean()] * len(Y_test)
baseline_MAE = mean_absolute_error(Y_test, baseline_prediction)

# 2.2 Building and training model LinearRegression
model = LinearRegression()
model.fit(X_train, Y_train)

# 2.3 Prediction result in Test Set (.predict) and calculate MAE
y_predict = model.predict(X_test)
model_MAE = mean_absolute_error(Y_test, y_predict)

# Evaluation MAE
print(f"1. Error of baseline (MAE): {baseline_MAE:.2f} Millions €")
print(f"2. Error of ML Model (MAE): {model_MAE:.2f} Millions €")
print(f"Model MAE is better that Baseline {((baseline_MAE - model_MAE) / baseline_MAE)*100:.2f}%")

1. Error of baseline (MAE): 35.90 Millions €
2. Error of ML Model (MAE): 20.01 Millions €
Model MAE is better that Baseline 44.27%


In [164]:
# Prediction

# 3.1 Validation weight (Feature importance / Coefficient)
coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient (Millions € per unit)": model.coef_.round(2)
})
print("<-----Influence of each skill on a player's value----->")
display(coef_df)

# 3.2 Use the model to predict the prices of new player from the new file .CSV
scouted = pd.read_csv("../data/scouted_players.csv")

scouted["prediction_value_millions"] = model.predict(scouted[feature_cols]).round(2)

print("\n=== Evaluation from future star players (Unseen Data) ===")
display(scouted[["name", "overall", "age", "pace", "shooting", "prediction_value_millions"]])

<-----Influence of each skill on a player's value----->


,Feature,Coefficient (Millions € per unit)
0,overall,-1.14
1,age,-6.40
2,pace,0.57
3,shooting,1.16



=== Evaluation from future star players (Unseen Data) ===


,name,overall,age,pace,shooting,prediction_value_millions
0,Lamine Yamal,84,17,85,78,158.29
1,Endrick,77,18,86,78,160.46
2,Arda Guler,78,19,74,75,142.58
3,Warren Zaire-Emery,80,18,76,68,139.71
4,Kobbie Mainoo,79,19,72,66,129.84
5,Alejandro Garnacho,80,20,86,76,141.90
6,Savinho,82,20,88,74,138.43
7,Evan Ferguson,77,20,75,79,142.54
